In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression,Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor,StackingRegressor


import warnings
warnings.filterwarnings('ignore')


In [36]:
df = pd.read_csv("bangladesh_student_performance.csv")
display(df.head())

,date,gender,age,address,famsize,Pstatus,M_Edu,F_Edu,M_Job,F_Job,relationship,smoker,tuition_fee,time_friends,ssc_result,hsc_result
0,29/04/2018,M,18,Rural,GT3,Together,3,2,At_home,Farmer,No,No,71672,4,4.22,3.72
1,29/04/2018,F,19,Rural,LE3,Apart,0,4,Other,Health,Yes,No,26085,5,3.47,2.62
2,29/04/2018,F,19,Rural,GT3,Together,0,3,Teacher,Services,No,No,40891,3,3.32,2.56
3,29/04/2018,F,19,Rural,LE3,Apart,2,3,At_home,Business,No,No,50600,2,4.57,4.17
4,29/04/2018,M,17,Rural,GT3,Together,1,1,At_home,Farmer,No,No,62458,2,4.50,3.94


In [37]:

from ydata_profiling import ProfileReport
profile = ProfileReport(df,title='Bangladesh Student Performance Prediction', explorative=True)
profile.to_file("bangladesh_student_performance_report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 16/16 [00:00<00:00, 306.35it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [38]:
df.columns

Index(['date', 'gender', 'age', 'address', 'famsize', 'Pstatus', 'M_Edu',
       'F_Edu', 'M_Job', 'F_Job', 'relationship', 'smoker', 'tuition_fee',
       'time_friends', 'ssc_result', 'hsc_result'],
      dtype='object')

In [39]:
df.columns = df.columns.str.strip().str.lower()
df.columns

Index(['date', 'gender', 'age', 'address', 'famsize', 'pstatus', 'm_edu',
       'f_edu', 'm_job', 'f_job', 'relationship', 'smoker', 'tuition_fee',
       'time_friends', 'ssc_result', 'hsc_result'],
      dtype='object')

In [40]:
df.drop(columns=['date'],inplace=True)

In [41]:

#^ correlation for numerical values
corr_target = df.select_dtypes(include=np.number).corr()['hsc_result'].sort_values(ascending=False)
corr_target


hsc_result      1.000000
ssc_result      0.950178
m_edu           0.063776
f_edu           0.054811
tuition_fee     0.038068
age            -0.009857
time_friends   -0.156356
Name: hsc_result, dtype: float64

In [47]:

#^ Separate X and y
X = df.drop(columns=['hsc_result'],axis=1) # means axis=1 for columns
y = df['hsc_result']



In [48]:

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  
    ('scaler', StandardScaler())
])

In [49]:
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [51]:

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((1614, 14), (404, 14), (1614,), (404,))

In [52]:

#& Base Learner
reg_lr =LinearRegression
reg_rf =RandomForestRegressor(n_estimators=100, random_state=42)
reg_gb =GradientBoostingRegressor(n_estimators=100, random_state=42)
